# Merge 2018-2019 Floors 3, 4, 5, and 7

This notebook is a focused version of the merge workflow. It only merges the raw CSV files for floors 3, 4, 5, and 7 across 2018 and 2019.

Scope:

- Years: `2018`, `2019`
- Floors: `3`, `4`, `5`, `7`
- Input folder: `Raw Data/`
- Output folder: `outputs/`
- Main output: `combined_wide_2018_2019_floors_3_4_5_7.csv.gz`

The merge is vertical concatenation. It keeps `year`, `floor`, and `source_file` columns so every row can be traced back to its original CSV file.


## 1. Import libraries and locate project folders


In [1]:
from pathlib import Path
from typing import Optional
import re

import pandas as pd

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

PROJECT_NAME = "Artificial Intelligence for Smart-Building Log Analysis"
YEARS = [2018, 2019]
FLOORS = [3, 4, 5, 7]


def find_project_dir(start: Optional[Path] = None) -> Path:
    """Find the project folder from the current notebook location or parent folders."""
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if path.name == PROJECT_NAME and (path / "Raw Data").exists():
            return path
        candidate = path / PROJECT_NAME
        if (candidate / "Raw Data").exists():
            return candidate
    raise FileNotFoundError(f"Could not find '{PROJECT_NAME}/Raw Data'.")


PROJECT_DIR = find_project_dir()
DATA_DIR = PROJECT_DIR / "Raw Data"
OUTPUT_DIR = PROJECT_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PROJECT_DIR, DATA_DIR, OUTPUT_DIR


(PosixPath('/Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis'),
 PosixPath('/Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/Raw Data'),
 PosixPath('/Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs'))

## 2. Select the required 8 source CSV files


In [2]:
source_files = [DATA_DIR / f"{year}Floor{floor}.csv" for year in YEARS for floor in FLOORS]
missing_files = [path for path in source_files if not path.exists()]

if missing_files:
    raise FileNotFoundError("Missing required CSV files:\n" + "\n".join(str(path) for path in missing_files))

source_file_table = pd.DataFrame({
    "year": [int(re.search(r"(20\d{2})", path.name).group(1)) for path in source_files],
    "floor": [int(re.search(r"Floor(\d+)", path.name).group(1)) for path in source_files],
    "file_name": [path.name for path in source_files],
    "file_path": [str(path) for path in source_files],
})

source_file_table


,year,floor,file_name,file_path
0,2018,3,2018Floor3.csv,/Users/yonglijiang/Documents/Adelaide Universi...
1,2018,4,2018Floor4.csv,/Users/yonglijiang/Documents/Adelaide Universi...
2,2018,5,2018Floor5.csv,/Users/yonglijiang/Documents/Adelaide Universi...
3,2018,7,2018Floor7.csv,/Users/yonglijiang/Documents/Adelaide Universi...
4,2019,3,2019Floor3.csv,/Users/yonglijiang/Documents/Adelaide Universi...
5,2019,4,2019Floor4.csv,/Users/yonglijiang/Documents/Adelaide Universi...
6,2019,5,2019Floor5.csv,/Users/yonglijiang/Documents/Adelaide Universi...
7,2019,7,2019Floor7.csv,/Users/yonglijiang/Documents/Adelaide Universi...


## 3. Check column counts and field consistency

Floors 3, 4, 5, and 7 should use the same 30 fields. This check prevents accidental merging by position when field names do not match.


In [3]:
headers = {}
for path in source_files:
    headers[path.name] = list(pd.read_csv(path, nrows=0).columns)

reference_file = source_files[0].name
reference_columns = headers[reference_file]

column_check = pd.DataFrame({
    "file_name": list(headers.keys()),
    "column_count": [len(cols) for cols in headers.values()],
    "same_columns_and_order_as_reference": [cols == reference_columns for cols in headers.values()],
})

COLUMN_CHECK_PATH = OUTPUT_DIR / "column_check_2018_2019_floors_3_4_5_7.csv"
column_check.to_csv(COLUMN_CHECK_PATH, index=False)

print("Reference file:", reference_file)
print("Reference column count:", len(reference_columns))
print("Column check saved to:", COLUMN_CHECK_PATH)

if not column_check["same_columns_and_order_as_reference"].all():
    mismatched = column_check.loc[~column_check["same_columns_and_order_as_reference"], "file_name"].tolist()
    raise ValueError(f"Column mismatch found in: {mismatched}")

column_check


Reference file: 2018Floor3.csv
Reference column count: 30
Column check saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/column_check_2018_2019_floors_3_4_5_7.csv


,file_name,column_count,same_columns_and_order_as_reference
0,2018Floor3.csv,30,True
1,2018Floor4.csv,30,True
2,2018Floor5.csv,30,True
3,2018Floor7.csv,30,True
4,2019Floor3.csv,30,True
5,2019Floor4.csv,30,True
6,2019Floor5.csv,30,True
7,2019Floor7.csv,30,True


## 4. Show the 30 columns used in the merge


In [4]:
pd.DataFrame({"column_index": range(1, len(reference_columns) + 1), "column_name": reference_columns})


,column_index,column_name
0,1,Date
1,2,z1_AC1(kW)
2,3,z1_AC2(kW)
3,4,z1_AC3(kW)
4,5,z1_AC4(kW)
5,6,z1_Light(kW)
6,7,z1_Plug(kW)
7,8,z1_S1(degC)
8,9,z1_S1(RH%)
9,10,z1_S1(lux)


## 5. Count rows before merging


In [5]:
row_counts = []
for path in source_files:
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        rows = sum(1 for _ in handle) - 1
    year = int(re.search(r"(20\d{2})", path.name).group(1))
    floor = int(re.search(r"Floor(\d+)", path.name).group(1))
    row_counts.append({"year": year, "floor": floor, "file_name": path.name, "row_count": rows})

row_count_table = pd.DataFrame(row_counts)
ROW_COUNT_PATH = OUTPUT_DIR / "row_counts_2018_2019_floors_3_4_5_7.csv"
row_count_table.to_csv(ROW_COUNT_PATH, index=False)

print("Expected merged rows:", f"{row_count_table['row_count'].sum():,}")
print("Row count summary saved to:", ROW_COUNT_PATH)

row_count_table


Expected merged rows: 3,162,240
Row count summary saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/row_counts_2018_2019_floors_3_4_5_7.csv


,year,floor,file_name,row_count
0,2018,3,2018Floor3.csv,264960
1,2018,4,2018Floor4.csv,264960
2,2018,5,2018Floor5.csv,264960
3,2018,7,2018Floor7.csv,264960
4,2019,3,2019Floor3.csv,525600
5,2019,4,2019Floor4.csv,525600
6,2019,5,2019Floor5.csv,525600
7,2019,7,2019Floor7.csv,525600


## 6. Merge the selected CSV files

This section reads each selected raw file, standardizes the `Date` column to datetime, adds traceability columns, and vertically concatenates the tables.


In [6]:
merged_parts = []

for path in source_files:
    year = int(re.search(r"(20\d{2})", path.name).group(1))
    floor = int(re.search(r"Floor(\d+)", path.name).group(1))

    df = pd.read_csv(path)
    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df.insert(0, "source_file", path.name)
    df.insert(0, "floor", floor)
    df.insert(0, "year", year)
    merged_parts.append(df)

combined = pd.concat(merged_parts, ignore_index=True)
combined = combined.sort_values(["year", "floor", "Date"]).reset_index(drop=True)

combined.shape


(3162240, 33)

## 7. Validate the merged table


In [7]:
validation_summary = {
    "rows": len(combined),
    "columns": combined.shape[1],
    "expected_rows_from_sources": int(row_count_table["row_count"].sum()),
    "invalid_datetime_rows": int(combined["Date"].isna().sum()),
    "duplicate_year_floor_date_rows": int(combined.duplicated(["year", "floor", "Date"]).sum()),
    "min_date": combined["Date"].min(),
    "max_date": combined["Date"].max(),
}

validation_summary_df = pd.DataFrame([validation_summary])
validation_summary_df


,rows,columns,expected_rows_from_sources,invalid_datetime_rows,duplicate_year_floor_date_rows,min_date,max_date
0,3162240,33,3162240,0,0,2018-07-01,2019-12-31 23:59:00


## 8. Save the merged output


In [8]:
OUTPUT_FILE = OUTPUT_DIR / "combined_wide_2018_2019_floors_3_4_5_7.csv.gz"
PREVIEW_FILE = OUTPUT_DIR / "combined_wide_2018_2019_floors_3_4_5_7_preview.csv"
SUMMARY_FILE = OUTPUT_DIR / "merge_summary_2018_2019_floors_3_4_5_7.csv"

combined.to_csv(OUTPUT_FILE, index=False, compression="gzip")
combined.head(1000).to_csv(PREVIEW_FILE, index=False)
validation_summary_df.to_csv(SUMMARY_FILE, index=False)

print("Merged output saved to:", OUTPUT_FILE)
print("Preview saved to:", PREVIEW_FILE)
print("Merge summary saved to:", SUMMARY_FILE)
print("Rows:", f"{len(combined):,}")
print("Columns:", combined.shape[1])


Merged output saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/combined_wide_2018_2019_floors_3_4_5_7.csv.gz
Preview saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/combined_wide_2018_2019_floors_3_4_5_7_preview.csv
Merge summary saved to: /Users/yonglijiang/Documents/Adelaide University/AU Course/AU Course - 2026 Semester 2/Industry Research Project/Artificial Intelligence for Smart-Building Log Analysis/outputs/merge_summary_2018_2019_floors_3_4_5_7.csv
Rows: 3,162,240
Columns: 33


## 9. Quick review


In [13]:
display(combined.head())
display(combined.groupby(["year", "floor"]).size().reset_index(name="rows"))
combined.shape

,year,floor,source_file,Date,z1_AC1(kW),z1_AC2(kW),z1_AC3(kW),z1_AC4(kW),z1_Light(kW),z1_Plug(kW),z1_S1(degC),z1_S1(RH%),z1_S1(lux),z2_AC1(kW),z2_Light(kW),z2_Plug(kW),z2_S1(degC),z2_S1(RH%),z2_S1(lux),z3_Light(kW),z3_Plug(kW),z4_AC1(kW),z4_Light(kW),z4_Plug(kW),z4_S1(degC),z4_S1(RH%),z4_S1(lux),z5_AC1(kW),z5_Light(kW),z5_Plug(kW),z5_S1(degC),z5_S1(RH%),z5_S1(lux)
0,2018,3,2018Floor3.csv,2018-07-01 00:00:00,0.0,0.0,0.0,0.07,0.0,0.03,28.44,60.67,0.0,0.0,0.0,0.03,27.67,63.37,0.0,0.01,0.26,0.0,0.0,0.02,27.69,65.66,0.0,0.0,0.0,0.02,26.88,71.16,0.0
1,2018,3,2018Floor3.csv,2018-07-01 00:01:00,0.0,0.0,0.0,0.07,0.0,0.03,28.44,60.67,0.0,0.0,0.0,0.03,27.67,63.37,0.0,0.02,0.37,0.0,0.0,0.02,27.69,65.65,0.0,0.0,0.0,0.02,26.88,71.16,0.0
2,2018,3,2018Floor3.csv,2018-07-01 00:02:00,0.0,0.0,0.0,0.07,0.0,0.03,28.45,60.67,0.0,0.0,0.0,0.03,27.67,63.37,0.0,0.01,0.32,0.0,0.0,0.02,27.69,65.63,0.0,0.0,0.0,0.02,26.88,71.18,0.0
3,2018,3,2018Floor3.csv,2018-07-01 00:03:00,0.0,0.0,0.0,0.07,0.0,0.03,28.45,60.66,0.0,0.0,0.0,0.03,27.67,63.36,0.0,0.01,0.26,0.0,0.0,0.02,NaN,NaN,NaN,0.0,0.0,0.02,26.88,71.19,0.0
4,2018,3,2018Floor3.csv,2018-07-01 00:04:00,0.0,0.0,0.0,0.07,0.0,0.03,28.45,60.66,0.0,0.0,0.0,0.03,27.67,63.36,0.0,0.02,0.37,0.0,0.0,0.02,NaN,NaN,NaN,0.0,0.0,0.02,26.88,71.21,0.0


,year,floor,rows
0,2018,3,264960
1,2018,4,264960
2,2018,5,264960
3,2018,7,264960
4,2019,3,525600
5,2019,4,525600
6,2019,5,525600
7,2019,7,525600


(3162240, 33)

## 10. Data cleaning and zone-level conversion

This section converts the selected wide floor-level tables into a modelling-friendly zone-level table and applies the first cleaning rules before EDA or rolling features.

Cleaning rules:

- Convert `Date` to a unified datetime field named `timestamp`.
- Sort by `floor`, `zone`, and `timestamp`.
- Remove duplicate records within each source file.
- Standardize measurement names and units.
- Preserve real zero values.
- Convert impossible values to missing values and record their counts.
- Resample each `floor + zone` group to 15-minute intervals.
- Add flags for missing or incomplete time windows.


In [ ]:
POWER_MIN_VALUE = 0
TEMPERATURE_MIN_C = 10
TEMPERATURE_MAX_C = 45
HUMIDITY_MIN_PCT = 0
HUMIDITY_MAX_PCT = 100
LUX_MIN_VALUE = 0
RESAMPLE_INTERVAL = "15min"
EXPECTED_MINUTES_PER_WINDOW = 15

POWER_COLUMNS = [col for col in reference_columns if col.endswith("(kW)")]
ZONE_IDS = sorted({int(match.group(1)) for col in reference_columns if (match := re.match(r"z(\d+)_", col))})

ZONE_IDS, POWER_COLUMNS[:5], len(POWER_COLUMNS)


## 11. Build a cleaned minute-level zone table


In [ ]:
def build_zone_minute_table(df: pd.DataFrame, year: int, floor: int, source_file: str) -> tuple[pd.DataFrame, dict, list[dict]]:
    working = df.copy()
    original_rows = len(working)

    working["timestamp"] = pd.to_datetime(working["Date"], errors="coerce")
    invalid_timestamp_rows = int(working["timestamp"].isna().sum())
    working = working.dropna(subset=["timestamp"])

    duplicate_timestamp_rows = int(working.duplicated(["timestamp"]).sum())
    working = working.drop_duplicates(["timestamp"], keep="first")
    working = working.sort_values("timestamp").reset_index(drop=True)

    measurement_columns = [col for col in working.columns if col.startswith("z")]
    for col in measurement_columns:
        working[col] = pd.to_numeric(working[col], errors="coerce")

    invalid_records = []

    for col in POWER_COLUMNS:
        if col in working.columns:
            invalid_mask = working[col] < POWER_MIN_VALUE
            invalid_records.append({
                "source_file": source_file,
                "year": year,
                "floor": floor,
                "column": col,
                "invalid_rule": "power_kw < 0",
                "invalid_count": int(invalid_mask.sum()),
            })
            working.loc[invalid_mask, col] = pd.NA

    range_rules = [
        ("(degC)", "temperature_c outside 10-45", lambda s: (s < TEMPERATURE_MIN_C) | (s > TEMPERATURE_MAX_C)),
        ("(RH%)", "humidity_pct outside 0-100", lambda s: (s < HUMIDITY_MIN_PCT) | (s > HUMIDITY_MAX_PCT)),
        ("(lux)", "lux < 0", lambda s: s < LUX_MIN_VALUE),
    ]

    for suffix, rule_name, rule_func in range_rules:
        for col in [c for c in measurement_columns if c.endswith(suffix)]:
            invalid_mask = rule_func(working[col])
            invalid_records.append({
                "source_file": source_file,
                "year": year,
                "floor": floor,
                "column": col,
                "invalid_rule": rule_name,
                "invalid_count": int(invalid_mask.sum()),
            })
            working.loc[invalid_mask, col] = pd.NA

    zone_frames = []
    for zone in ZONE_IDS:
        prefix = f"z{zone}_"
        ac_cols = [col for col in working.columns if re.fullmatch(rf"{prefix}AC\d+\(kW\)", col)]
        light_col = f"{prefix}Light(kW)"
        plug_col = f"{prefix}Plug(kW)"
        temp_col = f"{prefix}S1(degC)"
        humidity_col = f"{prefix}S1(RH%)"
        lux_col = f"{prefix}S1(lux)"

        zone_df = pd.DataFrame({
            "timestamp": working["timestamp"],
            "year": year,
            "floor": floor,
            "zone": zone,
            "source_file": source_file,
            "has_ac_meter": bool(ac_cols),
            "has_light_meter": light_col in working.columns,
            "has_plug_meter": plug_col in working.columns,
            "has_environment_sensor": all(col in working.columns for col in [temp_col, humidity_col, lux_col]),
        })

        missing_numeric = pd.Series(pd.NA, index=working.index, dtype="Float64")
        zone_df["ac_power_kw"] = working[ac_cols].sum(axis=1, min_count=1) if ac_cols else missing_numeric.copy()
        zone_df["light_power_kw"] = working[light_col] if light_col in working.columns else missing_numeric.copy()
        zone_df["plug_power_kw"] = working[plug_col] if plug_col in working.columns else missing_numeric.copy()
        zone_df["temperature_c"] = working[temp_col] if temp_col in working.columns else missing_numeric.copy()
        zone_df["humidity_pct"] = working[humidity_col] if humidity_col in working.columns else missing_numeric.copy()
        zone_df["lux"] = working[lux_col] if lux_col in working.columns else missing_numeric.copy()
        zone_df["total_power_kw"] = zone_df[["ac_power_kw", "light_power_kw", "plug_power_kw"]].sum(axis=1, min_count=1)

        zone_frames.append(zone_df)

    audit = {
        "source_file": source_file,
        "year": year,
        "floor": floor,
        "raw_rows": original_rows,
        "invalid_timestamp_rows": invalid_timestamp_rows,
        "duplicate_timestamp_rows_removed": duplicate_timestamp_rows,
        "clean_minute_rows": len(working),
        "zone_minute_rows": len(working) * len(ZONE_IDS),
    }

    return pd.concat(zone_frames, ignore_index=True), audit, invalid_records


## 12. Resample each floor-zone group to 15-minute intervals


In [ ]:
VALUE_COLUMNS = ["ac_power_kw", "light_power_kw", "plug_power_kw", "total_power_kw", "temperature_c", "humidity_pct", "lux"]
AVAILABILITY_COLUMNS = ["has_ac_meter", "has_light_meter", "has_plug_meter", "has_environment_sensor"]


def resample_zone_table(zone_minute: pd.DataFrame) -> pd.DataFrame:
    resampled_parts = []

    for (year, floor, zone), group in zone_minute.groupby(["year", "floor", "zone"], sort=True):
        group = group.sort_values("timestamp").set_index("timestamp")
        start = group.index.min().floor(RESAMPLE_INTERVAL)
        end = group.index.max().floor(RESAMPLE_INTERVAL)
        full_index = pd.date_range(start=start, end=end, freq=RESAMPLE_INTERVAL)

        value_mean = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).mean().reindex(full_index)
        value_min = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).min().reindex(full_index).add_suffix("_min")
        value_max = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).max().reindex(full_index).add_suffix("_max")
        valid_counts = group[VALUE_COLUMNS].resample(RESAMPLE_INTERVAL).count().reindex(full_index, fill_value=0).add_suffix("_valid_count")
        observed_minutes = group["source_file"].resample(RESAMPLE_INTERVAL).count().reindex(full_index, fill_value=0)

        out = pd.concat([value_mean, value_min, value_max, valid_counts], axis=1)
        out.insert(0, "timestamp", full_index)
        out.insert(1, "year", year)
        out.insert(2, "floor", floor)
        out.insert(3, "zone", zone)
        out["observed_minutes"] = observed_minutes.to_numpy()
        out["missing_time_flag"] = out["observed_minutes"] == 0
        out["incomplete_window_flag"] = out["observed_minutes"] < EXPECTED_MINUTES_PER_WINDOW

        for col in AVAILABILITY_COLUMNS:
            out[col] = bool(group[col].iloc[0])

        for col in VALUE_COLUMNS:
            out[f"{col}_missing_flag"] = out[col].isna()

        resampled_parts.append(out.reset_index(drop=True))

    cleaned = pd.concat(resampled_parts, ignore_index=True)
    cleaned = cleaned.sort_values(["floor", "zone", "timestamp"]).reset_index(drop=True)
    return cleaned


## 13. Run cleaning for selected floors


In [ ]:
zone_minute_parts = []
cleaning_audits = []
invalid_records_all = []

for path in source_files:
    year = int(re.search(r"(20\d{2})", path.name).group(1))
    floor = int(re.search(r"Floor(\d+)", path.name).group(1))
    raw_df = pd.read_csv(path)
    zone_minute, audit, invalid_records = build_zone_minute_table(raw_df, year, floor, path.name)
    zone_minute_parts.append(zone_minute)
    cleaning_audits.append(audit)
    invalid_records_all.extend(invalid_records)
    print(f"Cleaned {path.name}: {len(zone_minute):,} zone-minute rows")

zone_minute_all = pd.concat(zone_minute_parts, ignore_index=True)
cleaned_15min = resample_zone_table(zone_minute_all)

print("Minute-level zone rows:", f"{len(zone_minute_all):,}")
print("Cleaned 15-minute rows:", f"{len(cleaned_15min):,}")
print("Cleaned columns:", cleaned_15min.shape[1])

cleaned_15min.head()


## 14. Save cleaning outputs


In [ ]:
CLEANED_15MIN_FILE = OUTPUT_DIR / "cleaned_zone_15min_2018_2019_floors_3_4_5_7.csv.gz"
CLEANED_PREVIEW_FILE = OUTPUT_DIR / "cleaned_zone_15min_2018_2019_floors_3_4_5_7_preview.csv"
CLEANING_SUMMARY_FILE = OUTPUT_DIR / "cleaning_summary_2018_2019_floors_3_4_5_7.csv"
INVALID_SUMMARY_FILE = OUTPUT_DIR / "invalid_values_summary_2018_2019_floors_3_4_5_7.csv"
MISSING_SUMMARY_FILE = OUTPUT_DIR / "missing_summary_2018_2019_floors_3_4_5_7.csv"

cleaned_15min.to_csv(CLEANED_15MIN_FILE, index=False, compression="gzip")
cleaned_15min.head(1000).to_csv(CLEANED_PREVIEW_FILE, index=False)

cleaning_summary = pd.DataFrame(cleaning_audits)
invalid_summary = pd.DataFrame(invalid_records_all)
missing_summary = pd.DataFrame({
    "column": cleaned_15min.columns,
    "missing_count": [int(cleaned_15min[col].isna().sum()) for col in cleaned_15min.columns],
    "missing_pct": [float(cleaned_15min[col].isna().mean() * 100) for col in cleaned_15min.columns],
})

cleaning_summary.to_csv(CLEANING_SUMMARY_FILE, index=False)
invalid_summary.to_csv(INVALID_SUMMARY_FILE, index=False)
missing_summary.to_csv(MISSING_SUMMARY_FILE, index=False)

print("Cleaned 15-minute data:", CLEANED_15MIN_FILE)
print("Preview:", CLEANED_PREVIEW_FILE)
print("Cleaning summary:", CLEANING_SUMMARY_FILE)
print("Invalid value summary:", INVALID_SUMMARY_FILE)
print("Missing summary:", MISSING_SUMMARY_FILE)


## 15. Review cleaning results


In [ ]:
review = {
    "rows": len(cleaned_15min),
    "columns": cleaned_15min.shape[1],
    "floors": sorted(cleaned_15min["floor"].unique().tolist()),
    "zones": sorted(cleaned_15min["zone"].unique().tolist()),
    "min_timestamp": cleaned_15min["timestamp"].min(),
    "max_timestamp": cleaned_15min["timestamp"].max(),
    "duplicate_floor_zone_timestamp_rows": int(cleaned_15min.duplicated(["floor", "zone", "timestamp"]).sum()),
    "missing_time_windows": int(cleaned_15min["missing_time_flag"].sum()),
    "incomplete_15min_windows": int(cleaned_15min["incomplete_window_flag"].sum()),
    "invalid_values_detected": int(invalid_summary["invalid_count"].sum()),
}

review_df = pd.DataFrame([review])
display(review_df)
display(missing_summary.sort_values(["missing_pct", "missing_count"], ascending=False).head(20))
display(invalid_summary.query("invalid_count > 0"))
